In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

## Byte-Pair Encoding (BPE)

### Algorithm (training)
1. Split each word into characters + `</w>` end marker
2. Count all adjacent pairs across the corpus
3. Merge the most frequent pair into a single token
4. Repeat for `num_merges` iterations

In [26]:
class SimpleBPE:
    def __init__(self):
        self.merges = []

    def train(self, corpus, num_merges):
        vocab = {}
        for word in corpus:
            symbols = tuple(word) + ('</w>',)
            vocab[symbols] = vocab.get(symbols, 0) + 1
        self.merges = []
        for _ in range(num_merges):
            pairs = {}
            for word, freq in vocab.items():
                for i in range(len(word) - 1):
                    pair = (word[i], word[i + 1])
                    pairs[pair] = pairs.get(pair, 0) + freq
            if not pairs:
                break
            best = max(pairs, key=pairs.get)
            self.merges.append(best)
            new_vocab = {}
            for word, freq in vocab.items():
                new_word = []
                i = 0
                while i < len(word):
                    if i < len(word) - 1 and (word[i], word[i + 1]) == best:
                        new_word.append(word[i] + word[i + 1])
                        i += 2
                    else:
                        new_word.append(word[i])
                        i += 1
                new_vocab[tuple(new_word)] = freq
            vocab = new_vocab

    def encode(self, text):
        all_tokens = []
        for word in text.split():
            symbols = list(word) + ['</w>']
            for a, b in self.merges:
                i = 0
                while i < len(symbols) - 1:
                    if symbols[i] == a and symbols[i + 1] == b:
                        symbols = symbols[:i] + [a + b] + symbols[i + 2:]
                    else:
                        i += 1
            all_tokens.extend(symbols)
        return all_tokens

In [27]:

bpe = SimpleBPE()
bpe.train(['low', 'low', 'low', 'lower', 'newest', 'widest'], num_merges=10)
print('Merges:', bpe.merges)
print('Encode:', bpe.encode('low lower newest'))

Merges: [('l', 'o'), ('lo', 'w'), ('low', '</w>'), ('e', 's'), ('es', 't'), ('est', '</w>'), ('low', 'e'), ('lowe', 'r'), ('lower', '</w>'), ('n', 'e')]
Encode: ['low</w>', 'lower</w>', 'ne', 'w', 'est</w>']


## INT8 Quantized Linear


### Quantization (per-channel)
1. `scale = weight.abs().max(dim=1) / 127`
2. `weight_int8 = round(weight / scale).clamp(-128, 127).to(int8)`
3. Store as `register_buffer` (not trainable)
4. Forward: dequantize (`int8.float() * scale`) then matmul

In [28]:
class Int8Linear(nn.Module):
  def __init__(self, weight, bias = None):
    super().__init__()
    scale = weight.abs().amax(dim=1, keepdim=True) / 127.0
    self.register_buffer('weight_int8', torch.round(weight / scale + 1e-10).clamp(-128,127).to(torch.int8))
    self.register_buffer('scale', scale)
    self.bias = nn.Parameter(bias.clone()) if bias is not None else None
  def forward(self, x):
    w = self.weight_int8.float() * self.scale
    out = x @ w.T
    if self.bias is not None:
      out = out + self.bias
    return out

In [29]:
w = torch.randn(8, 4)
q = Int8Linear(w)
print('Output:', q(torch.randn(2, 4)).dtype)
print('Weight dtype:', q.weight_int8.dtype)

Output: torch.float32
Weight dtype: torch.int8


## DPO Loss

$$\mathcal{L}_{\text{DPO}} = -\log \sigma\Big(\beta \big[\log\frac{\pi(y_w)}{\text{ref}(y_w)} - \log\frac{\pi(y_l)}{\text{ref}(y_l)}\big]\Big)$$


In [30]:
def dpo_loss(policy_chosen_logps, policy_rejected_logps, ref_chosen_logps, ref_rejected_logps, beta = 0.1):
  chosen_rewards = beta * (policy_chosen_logps - ref_chosen_logps)
  rejected_rewards = beta * (policy_rejected_logps - ref_rejected_logps)
  return -F.logsigmoid(chosen_rewards - rejected_rewards).mean()

In [31]:

chosen = torch.tensor([0.0, 0.0])
rejected = torch.tensor([-5.0, -5.0])
ref_c = torch.tensor([-1.0, -1.0])
ref_r = torch.tensor([-1.0, -1.0])
print('Loss:', dpo_loss(chosen, rejected, ref_c, ref_r, beta=0.1).item())

Loss: 0.4740769863128662
